# Part 3: AI & Self-Service Data Platform
## Natural Language → SQL → Insights POC

**Objective:** Demonstrate a lightweight POC where a business user can query customer data using natural language.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import sqlite3
from pathlib import Path

from src.data import DataLoader, DataPreprocessor

---
## 1. Load Data into SQLite

In [2]:
DATA_PATH = Path("../data/Senior_Data_Scientist_Assessment_Data.xlsx")

loader = DataLoader(DATA_PATH)
raw = loader.load_all()
preprocessor = DataPreprocessor(raw)
cleaned = preprocessor.clean_all()

conn = sqlite3.connect(":memory:")
for name, df in cleaned.items():
    df.to_sql(name.lower(), conn, if_exists="replace", index=False)
print("Tables loaded:", [name.lower() for name in cleaned.keys()])

Tables loaded: ['customers', 'departments', 'users', 'products', 'leads', 'installations', 'accounts', 'sales']


## 2. Define Schema Context

In [3]:
SQL_SCHEMA = ""
for name, df in cleaned.items():
    cols = ", ".join([f"{c} ({dtype})" for c, dtype in df.dtypes.items()])
    SQL_SCHEMA += f"Table: {name.lower()} | Columns: {cols}\n"
    for col in df.select_dtypes(include="object").columns[:3]:
        vals = df[col].dropna().unique()[:5]
        SQL_SCHEMA += f"  Sample {col} values: {list(vals)}\n"

print(SQL_SCHEMA[:1200])

Table: customers | Columns: id (object), region (object), gender (object), location (object), date_of_birth (object), latitude (float64), longitude (float64), created_at (object), updated_at (object)
  Sample id values: ['CUST-10000', 'CUST-10001', 'CUST-10002', 'CUST-10003', 'CUST-10004']
  Sample region values: ['uganda', 'civ', 'kenya']
  Sample gender values: ['Female', 'Male']
Table: departments | Columns: id (object), name (object), created_at (object), updated_at (object)
  Sample id values: ['D100', 'D101', 'D102', 'D103', 'D104']
  Sample name values: ['Sales', 'Engineering', 'Finance', 'Logistics', 'Customer Service']
  Sample created_at values: ['2021-01-01', '2021-03-29', '2021-01-03', '2021-02-26', '2021-05-02']
Table: users | Columns: id (object), name (object), departmentid (object), created_at (object), updated_at (object)
  Sample id values: ['U1000', 'U1001', 'U1002', 'U1003', 'U1004']
  Sample name values: ['Mary Moyo', 'David Kone', 'Mary Ouma', 'Sarah Koffi', 'Mose

## 3. Text-to-SQL via LLM

Using a prompt template + an LLM (OpenAI / Claude / local model) to translate natural language questions into SQL queries.

**Architecture:**
```
User Query (NL) → LLM → SQL → SQLite → Result → LLM → NL Answer + Chart
                                  ↑
                           Validation Layer
```

In [4]:
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
import os

load_dotenv(find_dotenv())
client = OpenAI()

def ask(question: str) -> str:
    prompt = f"""Given the following SQLite schema:\n{SQL_SCHEMA}\n
    Generate a SQL query to answer: {question}\n
    Return only the SQL query, no explanation."""
    response = client.chat.completions.create(
        model="gpt-4", messages=[{"role": "user", "content": prompt}], temperature=0
    )
    sql = response.choices[0].message.content.strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()
    try:
        result = pd.read_sql(sql, conn)
        return result.to_string(index=False)
    except Exception as e:
        return f"Query error: {e}"

def answer(question: str) -> str:
    query_result = ask(question)
    prompt = f"""Question: {question}\n
    Query result:\n{query_result}\n
    Answer in plain English with a business interpretation."""
    response = client.chat.completions.create(
        model="gpt-4", messages=[{"role": "user", "content": prompt}], temperature=0
    )
    return response.choices[0].message.content

# Example usage:
print(answer("Which country has the highest number of accounts in arrears?"))


The country with the highest number of accounts in arrears is Côte d'Ivoire (abbreviated as CIV), with a total of 205 accounts. This suggests that there is a significant number of individuals or businesses in Côte d'Ivoire who are behind on their payments, which could indicate financial difficulties within the country.


## 4. Lightweight Streamlit Alternative

For a fully interactive POC, wrap the above logic in a Streamlit app:

```python
import streamlit as st
st.title("SunCulture Data Assistant")
question = st.text_input("Ask a question about your data")
if question:
    response = answer(question)
    st.write(response)
```

**Tools used:** LangChain, OpenAI/Claude, SQLite, Streamlit, Plotly

**Key considerations:**
- Validation layer prevents destructive SQL (DDL/DML filtering)
- RBAC for data governance — which users can query which tables
- Confidence scores and SQL transparency for trust calibration
- Feedback loop: thumbs-up/down on answers → fine-tune prompt templates